# Hotel cancellation model: training pipeline

Predicts whether a hotel booking will be canceled, so a revenue manager can
decide how many extra rooms to sell (overbooking). The same logistic
regression is trained three ways: scikit-learn, a manual PyTorch loop, and
the standard `torch.nn.Module` + `torch.optim` workflow.

Run start to finish with `uv run python main.py train`.

## 1. Load the data

The CSV is a committed copy of the Kaggle dataset (see
`scripts/fetch_data.py`), so this runs offline and without Kaggle
credentials.

In [ ]:
from pathlib import Path

import pandas as pd

from pipeline import clean

DATA_PATH = Path("data/hotel_reservations.csv")

raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)

## 2. Clean

`clean` lives in `pipeline.py` because the app must apply exactly the same
fixes to new bookings. It fixes the invalid 29 Feb 2018 dates, adds quality
flags, caps implausible child counts, and builds the target `y`
(1 = canceled).

In [ ]:
df = clean(raw)

print("Rows moved from 29 Feb to 1 Mar:", df["invalid_date"].sum())
print("Zero-night bookings:            ", df["zero_nights"].sum())
print("Free rooms:                     ", df["is_free"].sum())
print("Max children after capping:     ", df["no_of_children"].max())
print("Missing arrival dates:          ", df["arrival"].isna().sum())
print("Overall cancellation rate:      ", round(df["y"].mean(), 3))

## 3. Time-based split

The model will be trained on past bookings and used on future ones, so the
split follows time: a random split would let the model see later bookings
during training and make the test score look better than it would be in
real use. Cancellation rates also change a lot over time, which a random
split would hide.

- Train: arrivals up to 30 Jun 2018 (fit the models)
- Validation: Jul to Sep 2018 (choose learning rate, regularization, threshold)
- Test: Oct to Dec 2018 (final score, used once)

In [ ]:
VAL_START = pd.Timestamp("2018-07-01")
TEST_START = pd.Timestamp("2018-10-01")

train_df = df[df["arrival"] < VAL_START]
val_df = df[(df["arrival"] >= VAL_START) & (df["arrival"] < TEST_START)]
test_df = df[df["arrival"] >= TEST_START]

assert len(train_df) + len(val_df) + len(test_df) == len(df)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(
        f"{name:<11} {part['arrival'].min():%Y-%m-%d} to {part['arrival'].max():%Y-%m-%d}"
        f"   rows = {len(part):>6,}   cancel rate = {part['y'].mean():.3f}"
    )